# Searching

This section builds search methods from simple to efficient, then benchmarks them.

**Learning Goals**
- Implement linear and binary search correctly
- Use hash-based lookup for repeated membership checks
- Compare runtime trends with repeated timing
- Choose a search strategy from data constraints

In [5]:
from time import perf_counter
import random

## Linear Search

Linear search is the simplest search strategy: it scans every element from left to right, one at a time, until the target is found or the list is exhausted. Because no prior knowledge about the data's order is assumed, it works on any collection — sorted or not.

| Case | Complexity | When it occurs |
|---|---|---|
| Best | $O(1)$ | Target is the first element |
| Average | $O(n)$ | Target is somewhere in the middle |
| Worst | $O(n)$ | Target is the last element or absent |

- Works on unsorted data
- Best when data is small or only one lookup is needed; for repeated lookups on large data, a `set` is faster

In [6]:
def linear_search(items, target):
    """Return index of target in items, or -1 if not found."""
    for i, value in enumerate(items):
        if value == target:
            return i
    return -1

sample = [7, 4, 9, 1, 3]
print(linear_search(sample, 9))   # 2
print(linear_search(sample, 8))   # -1

2
-1


In [3]:
### Exercise: Find Minimum Value
#   1. Write `min_value(nums)` that scans through the list in a single pass.
#   2. Return `None` for an empty list.
#   3. Return the smallest value otherwise (do not use the built-in `min`).
### Your code starts here.




### Your code ends here.

In [6]:
### Solution

def min_value(nums):
    if not nums:
        return None

    current_min = nums[0]
    for x in nums[1:]:
        if x < current_min:
            current_min = x
    return current_min

> **Next:** If data can be kept sorted and repeated lookups are needed, binary search is usually the better choice.

## Binary Search

Binary search exploits sorted order to eliminate half the remaining candidates with each comparison — similar to finding a name in a phone book by opening to the middle page rather than starting from page one.

The algorithm maintains a window `[left, right]`. At each step it checks the midpoint `mid`: if the target equals `items[mid]`, the search is done; if the target is smaller, the right half is discarded (`right = mid - 1`); if larger, the left half is discarded (`left = mid + 1`). This halves the search space every iteration, giving $O(\log n)$ runtime — a list with one million elements needs at most 20 comparisons.

**Why sorted input is required:** the decision to discard the left or right half only makes sense when elements are in order. On unsorted data, knowing the midpoint value tells you nothing about which half might contain the target.

**Index semantics:** `binary_search()` returns an index into the *sorted* list passed to it, not into the original unsorted list. When the position in the original list is needed, use `linear_search()` instead.

| Case | Complexity |
|---|---|
| Best | $O(1)$ — target is the midpoint on the first check |
| Worst / Average | $O(\log n)$ |

### Iterative Implementation

In [7]:
def binary_search(sorted_items, target):
    """Return index of target in sorted_items, or -1 if not found.

    Note: the returned index is a position in *sorted_items*, not the original
    unsorted list. Use linear_search when you need the original-list position.
    """
    left, right = 0, len(sorted_items) - 1

    while left <= right:
        mid = (left + right) // 2
        guess = sorted_items[mid]

        if guess == target:
            return mid
        if guess < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1

sorted_sample = [1, 3, 4, 7, 9, 14, 20]
print(binary_search(sorted_sample, 14))  # 5
print(binary_search(sorted_sample, 2))   # -1

5
-1


## The `bisect` Module

For production code, use `bisect` instead of re-implementing binary search.
It operates on sorted lists and is written in C — fast and already tested.

| Function | What it does |
|---|---|
| `bisect_left(a, x)` | Leftmost index where `x` can be inserted to keep `a` sorted |
| `bisect_right(a, x)` | Rightmost insertion index (alias: `bisect.bisect`) |
| `insort(a, x)` | Inserts `x` into `a` at its sorted position (in-place) |

The `bisect_search()` pattern shown below — calling `bisect_left()` and then verifying that the element at the returned index actually equals the target — is the standard production idiom for searching a sorted list. It is more reliable than re-implementing binary search from scratch and faster than `list.index()`, which performs a linear scan.

> **Note:** `insort` keeps a list sorted after each insertion — compare this with Sorting where the full list is sorted from scratch. For small incremental updates `insort` is more efficient; for bulk data, a full sort wins.

In [1]:
import bisect

data = [1, 3, 4, 7, 9, 14, 20]

# Locate an existing value
idx = bisect.bisect_left(data, 7)
print(f"bisect_left(7)  → index {idx}, value {data[idx]}")   # 3, value 7

# Find insertion point for a missing value
print(f"bisect_left(5)  → index {bisect.bisect_left(data, 5)}")   # 3
print(f"bisect_right(5) → index {bisect.bisect_right(data, 5)}")  # 3 (same when absent)

# Insert maintaining sorted order
bisect.insort(data, 5)
print(f"after insort(5): {data}")

# Practical search pattern: locate then verify
def bisect_search(sorted_items, target):
    idx = bisect.bisect_left(sorted_items, target)
    if idx < len(sorted_items) and sorted_items[idx] == target:
        return idx
    return -1

nums = [1, 3, 4, 7, 9, 14, 20]
print(bisect_search(nums, 9))   # 4
print(bisect_search(nums, 6))   # -1


bisect_left(7)  → index 3, value 7
bisect_left(5)  → index 3
bisect_right(5) → index 3
after insort(5): [1, 3, 4, 5, 7, 9, 14, 20]
4
-1


## Hash-Based Lookup

When the task is membership testing, a `set` or `dict` is often the most practical choice.
Average lookup is $O(1)$ and independent of element order.

Chapter 08 introduced the list-vs-dict/set lookup trade-off and showed that membership testing in a `set` is significantly faster than scanning a `list`. Here, that $O(1)$ option is placed alongside linear ($O(n)$) and binary ($O(\log n)$) search to compare all three strategies directly.

In [8]:
random.seed(42)
numbers = [random.randint(1, 20_000) for _ in range(15_000)]
lookup_set = set(numbers)

target = numbers[len(numbers) // 2]
missing = 999_999

print('target in list:', target in numbers)
print('target in set :', target in lookup_set)
print('missing in list:', missing in numbers)
print('missing in set :', missing in lookup_set)

target in list: True
target in set : True
missing in list: False
missing in set : False


## Correctness Checks

Before timing, confirm that each function handles the edge cases most likely to expose bugs: an empty input, a single-element list (both target present and absent), duplicate values where the returned index matters, and a target at the very end (worst case for linear search). Using `assert` rather than just printing means any failure raises an immediate error, making bugs impossible to overlook in benchmark output.

For duplicate values, be explicit about the contract: `binary_search()` only guarantees that it returns *a* matching index, while `binary_search_first()` later tightens that rule to the leftmost match.

In [8]:
tests = [
    ([], 5),
    ([10], 10),
    ([10], 5),
    ([1, 2, 2, 2, 3], 2),
    (list(range(100)), 99),
]

for arr, target in tests:
    lin_idx = linear_search(arr, target)
    arr_sorted = sorted(arr)
    bin_idx = binary_search(arr_sorted, target)

    # verify hash-based lookup agrees with membership result
    assert (target in set(arr)) == (target in arr)

    if target in arr:
        assert lin_idx != -1
        assert arr[lin_idx] == target
        assert bin_idx != -1
        assert arr_sorted[bin_idx] == target
    else:
        assert lin_idx == -1
        assert bin_idx == -1

print('All checks passed.')

All checks passed.


## Timing Comparison

Two workloads are benchmarked to reflect the two most common real-world scenarios:

- **One lookup:** the full cost including any preprocessing (sorting or building a `set`) is timed.
- **Many repeated lookups:** preprocessing is performed once inside the workflow function and reused across all queries.

Comparing these two scenarios reveals when preprocessing is worthwhile: if only one query is needed, the setup cost may dominate; if many queries follow, it is often amortized to near zero.

**Why wrapper functions are used.** `linear_search` and `binary_search` are passed directly to `average_runtime` because their signatures match `fn(data, target)`. Set membership (`target in lookup_set`) is an expression, not a callable with that signature, so it is wrapped in a named function that also includes the construction cost. This ensures each timing measurement reflects the same full workflow for every strategy.

We use `time.perf_counter()` here for higher-resolution benchmarks. The coarser `time.time()` from earlier chapters is sufficient for rough timing but less reliable for short operations.

In [9]:
def average_runtime(fn, *args, repeats=30):
    total = 0.0
    for _ in range(repeats):
        start = perf_counter()
        fn(*args)
        total += perf_counter() - start
    return total / repeats


def sort_then_binary_once(data, target):
    sorted_data = sorted(data)
    return binary_search(sorted_data, target)


def build_set_once(data, target):
    lookup_set = set(data)
    return target in lookup_set


def linear_many(data, targets):
    return sum(linear_search(data, target) != -1 for target in targets)


def sort_then_binary_many(data, targets):
    sorted_data = sorted(data)
    return sum(binary_search(sorted_data, target) != -1 for target in targets)


def build_set_many(data, targets):
    lookup_set = set(data)
    return sum(target in lookup_set for target in targets)


sizes = [10_000, 50_000, 100_000, 200_000]
print('One lookup: includes preprocessing cost')
print(f"{'n':>10} {'linear total':>15} {'sort+binary':>15} {'build+set':>15}")

for n in sizes:
    data = list(range(n))
    target = n - 1

    t_linear_one = average_runtime(linear_search, data, target)
    t_binary_one = average_runtime(sort_then_binary_once, data, target)
    t_set_one = average_runtime(build_set_once, data, target)

    print(f"{n:>10} {t_linear_one:>15.8f} {t_binary_one:>15.8f} {t_set_one:>15.8f}")

print()
print('Many lookups: preprocessing is paid once and reused')
print(f"{'n':>10} {'linear total':>15} {'sort+binary':>15} {'build+set':>15}")

for n in sizes:
    data = list(range(n))
    targets = [0, n // 4, n // 2, (3 * n) // 4, n - 1] * 20

    t_linear_many = average_runtime(linear_many, data, targets)
    t_binary_many = average_runtime(sort_then_binary_many, data, targets)
    t_set_many = average_runtime(build_set_many, data, targets)

    print(f"{n:>10} {t_linear_many:>15.8f} {t_binary_many:>15.8f} {t_set_many:>15.8f}")

One lookup: includes preprocessing cost
         n    linear total     sort+binary       build+set
     10000      0.00016727      0.00004033      0.00005157
     50000      0.00078375      0.00017800      0.00023957
    100000      0.00173296      0.00036697      0.00054247
    200000      0.00349816      0.00067281      0.00118613

Many lookups: preprocessing is paid once and reused
         n    linear total     sort+binary       build+set
     10000      0.00868556      0.00014242      0.00005333
     50000      0.04111158      0.00025918      0.00023597
    100000      0.08626271      0.00044275      0.00052760
    200000      0.15434788      0.00077229      0.00113295


In [10]:
### Exercise: Binary Search First
#   1. Implement `binary_search_first(sorted_items, target)`.
#   2. Return the first (leftmost) index of `target` when duplicates exist.
#   3. Return `-1` when the target is absent.
#   4. Validate with at least: empty input, all duplicates, target missing.
### Your code starts here.




### Your code ends here.

In [3]:
### Solution

def binary_search_first(sorted_items, target):
    left, right = 0, len(sorted_items) - 1
    result = -1

    while left <= right:
        mid = (left + right) // 2
        guess = sorted_items[mid]

        if guess == target:
            result = mid
            right = mid - 1
        elif guess < target:
            left = mid + 1
        else:
            right = mid - 1

    return result

dup_data = [1, 2, 2, 2, 4, 4, 5]
assert binary_search_first([], 2) == -1
assert binary_search_first([2, 2, 2], 2) == 0
assert binary_search_first(dup_data, 2) == 1
assert binary_search_first(dup_data, 4) == 4
assert binary_search_first(dup_data, 3) == -1

print(binary_search_first(dup_data, 2))  # expected: 1
print(binary_search_first(dup_data, 4))  # expected: 4
print(binary_search_first(dup_data, 3))  # expected: -1
print('binary_search_first checks passed.')

1
4
-1
binary_search_first checks passed.


## Search Strategy Summary

No single search method is optimal in all cases. The appropriate choice depends on the characteristics of the problem, including whether the data is sorted, how many lookups will be performed, and the size of the dataset.

| Strategy | Time | Needs sorted data? | Best for |
|---|---|---|---|
| Linear search | $O(n)$ | No | Small data; one-off lookup; unsorted input |
| Binary search | $O(\log n)$ | Yes | Large sorted data; repeated lookups |
| `bisect` module | $O(\log n)$ | Yes | Production code; sorted list; insertions too |
| `set` / `dict` lookup | $O(1)$ avg | No | Repeated membership tests; large data |

**Cost of sorting.** Binary search and `bisect` both require sorted input. Sorting requires $O(n \log n)$ time, so when only one or two lookups are needed, sorting first is often not the most efficient approach. In such cases, linear search on the original unsorted list may have a lower total cost. Whether sorting is worthwhile depends on both $n$ and the number of subsequent lookups.

**Pre-processing trade-off.** Converting a list to a `set` requires $O(n)$ time and $O(n)$ additional space, but each later membership test is $O(1)$ on average. When many membership tests are performed on the same collection, it is often advantageous to pay the conversion cost once and reuse the resulting `set`.

**Practical decision guide:**
- One lookup, unsorted → `in list` or linear search
- One lookup, already sorted → `bisect_search` or binary search
- Many lookups, order doesn't matter → build a `set` once, then use `in set`
- Many lookups, sorted order must be maintained (insertions too) → `bisect.insort` + `bisect_search`

In summary, the most appropriate strategy is determined by the total cost of the full workflow, not merely by the cost of a single lookup operation.